# EcoSignal Adaptive Field Sampling Planner — Deployment v3

**New deployment copy of your existing v2 planner. Your original notebook is preserved unchanged inside `reference/`.** Run this notebook in Google Colab with **Runtime → Run all**. The default mode builds real candidate geometry from the verified numeric GeoRisk coverage, then retrieves actual OpenStreetMap screening. This can take several minutes; an OSM failure produces an explicit unready package, never fake sites.

Outputs: a lightweight **Render backend ZIP**, a **Sites integration ZIP**, candidate/source manifests, local software-test results, and a preparation report. You still deploy the ZIP in Render and apply the adapter to the latest Site source. Running this notebook does not publish or modify EcoSignal.

### Explicit method changes

The current planner uses **relative 2024 terrain susceptibility, retake/review need, coverage gaps, and site-type diversity**. It retires the old mixed lichen-risk score and unsupported chlorosis/nitrophyte/growth-form pollution ordering. The default profile retains the ratios of the original surviving terms: terrain **37⅓%**, review **32%**, coverage gap **24%**, diversity **6⅔%**; historical-change/control terms are explicitly inactive. These are **planner-priority weights**, separate from EcoSignal's unified environmental indicator (35% satellite, 15% AQI, 25% visible coverage, 25% terrain; growth form unweighted). Neither set was statistically optimized.

An optional six-term historical-change profile is supported only with genuine observed DEM-change values, periods, normalization, and a source hash for **every candidate**. No future terrain projection is used. No absent component is quietly filled or renormalized row by row.

**No real dated, confirmed lichen records or independent ground truth are included.** The 24 embedded fixtures test real terrain numbers only. Synthetic adaptation tests remain test code and cannot become production observations. Planner priority is not environmental health, calibrated probability, model confidence, or scientific validation.


In [ ]:
# 1. Lightweight runtime dependencies. Python 3.11+ recommended.
import os, sys, subprocess, importlib.metadata
NOTEBOOK_TEST = os.environ.get("ECOSIGNAL_NOTEBOOK_TEST") == "1"
PINS = {"fastapi":"0.162.0", "pydantic":"2.14.1", "uvicorn":"0.54.0", "numpy":"2.3.5", "httpx":"0.28.1"}
if not NOTEBOOK_TEST:
    needed=[]
    for package, version in PINS.items():
        try: installed=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: installed=None
        if installed != version: needed.append(f"{package}=={version}")
    if needed: subprocess.check_call([sys.executable,"-m","pip","install","-q",*needed])
print("Runtime dependencies ready; no TensorFlow or .keras models are required.")


In [ ]:
# 2. Settings: edit only when you want a different input path/profile.
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
MODE = "bootstrap_osm"  # bootstrap_osm | import_csv | verify_only
GRID_SPACING_M = 240  # full coverage, bounded pool; different from old 30 m cropped frame
INPUT_CANDIDATES_CSV = ""  # import_csv: FULL access-screened pool, not top-10 output
SCREENING_SOURCE = ""  # import_csv: actual OSM/screening source description
SCREENING_DATE = ""    # import_csv: actual timezone-qualified retrieval time, e.g. ...Z
OBSERVATIONS_JSON = ""  # optional real EcoSignal-site-inputs.json / EcoSignal-field-records.json
PROFILE = "current_georisk_review_v1"
HISTORICAL_CHANGE_PROVENANCE = None  # see README for optional complete historical profile
RUN_LIVE_RENDER = False
RENDER_BASE_URL = ""  # your NEW planner service URL, not image-analysis backend URL
DOWNLOAD_OUTPUTS = IN_COLAB
WORK = Path("/content/EcoSignal_Adaptive_Planner_v3" if IN_COLAB else "EcoSignal_Adaptive_Planner_v3").resolve()
if NOTEBOOK_TEST:
    MODE="verify_only"; DOWNLOAD_OUTPUTS=False
    WORK=Path(os.environ.get("ECOSIGNAL_NOTEBOOK_TEST_DIR",str(WORK))).resolve()
if MODE not in {"bootstrap_osm","import_csv","verify_only"}: raise ValueError("Unknown preparation mode")
WORK.mkdir(parents=True,exist_ok=True)
print("Working directory:",WORK,"| Mode:",MODE)


### Embedded source and exact numerical reference

This notebook is self-contained. Its embedded archive includes the original v2 notebook, revised selector and OSM validation source; the 2024 numerical 20 m GeoRisk grid, fixed full sampled-cohort CDF, model/DEM hashes and 24 terrain fixtures; and the new runtime, tests and Sites adapter. No credentials, device observations or photos are included. The grid is sampled numerical model output, **not PNG-derived**, not a 20 m area mean, and not an exact arbitrary-location 1 m prediction. Reference version: `georisk-2024-native-features-20m-sampling-v1`.


In [ ]:
# 3. Verify and unpack embedded sources. Original source files are reference-only.
import base64, hashlib, io, json, zipfile
PAYLOAD_SHA256 = "b233b05d0b65d8cfac2f52fa6ce9c759fe40cdf953dadbf115e4398437a467c1"
PAYLOAD_B64 = """..."""
raw=base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256,"Embedded archive checksum mismatch"
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert z.testzip() is None,"Embedded archive CRC failure"
    for info in z.infolist():
        dest=(WORK/info.filename).resolve()
        if not dest.is_relative_to(WORK): raise ValueError("Unsafe archive path")
        dest.parent.mkdir(parents=True,exist_ok=True)
        dest.write_bytes(z.read(info))
if str(WORK) not in sys.path: sys.path.insert(0,str(WORK))
OUTPUT=WORK/"outputs"; OUTPUT.mkdir(exist_ok=True)
print("Verified source archive:",PAYLOAD_SHA256)
print("Original source hashes:",json.loads((WORK/"reference/original_source_hashes.json").read_text()))


In [ ]:
# 4. Verify numerical terrain data and every fixed-CDF entry before using them.
from prepare_candidates import NumericGeoRisk, prepare
GEO = NumericGeoRisk(WORK/"reference/georisk")
NUMERIC_EVIDENCE = GEO.check_fixtures(WORK/"reference/georisk/fixtures.json")
print(json.dumps(NUMERIC_EVIDENCE,indent=2))
print("Coverage WGS84 [west,south,east,north]:",GEO.manifest["bounds_wgs84"])
print("Outside coverage/nodata yields unavailable; no interpolation across gaps.")


### Candidate preparation and optional real observations

**Default `bootstrap_osm`:** creates new candidates within actual numeric coverage and applies the original OSMnx 2.1 road/building/water/land-use/access checks. It does not remove locations based on old unverified lichen points. Candidate grid spacing defaults to 240 m for full coverage; selected locations remain at least 120 m apart. OSM labels are not permission or evidence that a tree/lichen exists. Field verification remains necessary.

**`import_csv`:** enter the full previously screened candidate CSV and the actual source/retrieval date in Settings. Do not use just the top 10 recommendations. The importer requires explicit safety flags and road/rail/crop-edge distances. Old `risk_score`, `uncertainty_score`, stress and priority fields are discarded; terrain values are resampled numerically. The optional six-term profile additionally needs `recent_change_score` in 0–1 for every row plus documented observed-change provenance. Missing historical data does not block the default explicit profile.

**Real observations:** export **Observations → Evidence & exports → Export notebook site inputs** in EcoSignal and use the JSON companion. The native metadata includes distinct observation/site IDs, confirmed location, actual capture and measurement times, attribution/model version, coverage and confirmation. A latitude/longitude/coverage-only CSV cannot establish these facts. No record is included by default, and missing confirmation never becomes true automatically. Accepted biological coverage can improve sampling coverage even if its unified environmental score is incomplete.

To upload in Colab if needed, run `from google.colab import files; files.upload()` in a new cell, then put the uploaded filename in Settings. Avoid uploading photographs or secrets to this planner.


In [ ]:
# 5. Heavy preparation stays in Colab, outside the Render runtime.
from datetime import datetime, timezone
from eco_planner.engine import Config, validate_pool
DATA_READY=False; CANDIDATES=[]; CANDIDATE_MANIFEST=None
PREPARATION={"status":"not_started","real_candidate_rows":0}
config=json.loads((WORK/"config.json").read_text());config["profile"]=PROFILE
CONFIG=Config(**config).validate()
(WORK/"config.json").write_text(json.dumps(config,indent=2))
try:
    if MODE=="bootstrap_osm":
        subprocess.check_call([sys.executable,"-m","pip","install","-q","osmnx==2.1.0","geopandas>=1,<2","pandas>=2.2,<3"])
        from prepare_osm import screen
        source_path,osm_meta=screen(GEO,WORK/"preparation",spacing_m=GRID_SPACING_M)
        source_description=osm_meta["source"];source_date=osm_meta["retrieved_at"]
    elif MODE=="import_csv":
        source_path=Path(INPUT_CANDIDATES_CSV).expanduser()
        if not INPUT_CANDIDATES_CSV or not source_path.is_file(): raise ValueError("Set an existing full candidate CSV path")
        if not SCREENING_SOURCE.strip() or not SCREENING_DATE.strip(): raise ValueError("Supply the actual screening source and timezone-qualified retrieval date")
        source_description=SCREENING_SOURCE;source_date=SCREENING_DATE;osm_meta=None
    else:
        source_path=None;osm_meta=None
    if source_path is not None:
        CANDIDATES,CANDIDATE_MANIFEST=prepare(source_path,GEO,WORK/"data",screening_source=source_description,screening_date=source_date,historical_change_provenance=HISTORICAL_CHANGE_PROVENANCE)
        if osm_meta: CANDIDATE_MANIFEST["osm_screening"]=osm_meta
        validate_pool(CANDIDATES,CANDIDATE_MANIFEST,CONFIG)
        if CANDIDATE_MANIFEST.get("evidence_kind")=="synthetic_test": raise ValueError("Synthetic pool cannot be packaged as real candidates")
        (WORK/"data/manifest.json").write_text(json.dumps(CANDIDATE_MANIFEST,indent=2))
        DATA_READY=bool(CANDIDATES)
        PREPARATION={"status":"ready" if DATA_READY else "no_rows_passed_screening","real_candidate_rows":len(CANDIDATES),"source_rows":CANDIDATE_MANIFEST["source_rows"],"exclusions":CANDIDATE_MANIFEST["exclusions"]}
    else:
        PREPARATION={"status":"verification_only_no_candidate_preparation","real_candidate_rows":0}
except Exception as exc:
    DATA_READY=False
    PREPARATION={"status":"needs_real_candidate_preparation","real_candidate_rows":0,"error":str(exc),"retry":"Fix the real input or OSM retrieval and rerun this cell. No stale data will be bundled."}
print(json.dumps(PREPARATION,indent=2))


In [ ]:
# 6. Use only actual eligible site records; otherwise report candidate-only planning.
from eco_planner.engine import observations_from_document, rank_candidates
OBSERVATIONS=[]; PLAN=None
if OBSERVATIONS_JSON:
    OBSERVATIONS=observations_from_document(json.loads(Path(OBSERVATIONS_JSON).read_text()))
if DATA_READY:
    PLAN=rank_candidates(CANDIDATES,OBSERVATIONS,CANDIDATE_MANIFEST,CONFIG)
    (OUTPUT/"real_plan.json").write_text(json.dumps(PLAN,indent=2,allow_nan=False))
    print("Plan status:",PLAN["status"],"| Counts:",PLAN["counts"])
    for row in PLAN["recommendations"]:
        print(row["rank"],row["candidate_id"],round(row["priority_score"],2),row["objective"],row["access_status"])
    print("Rejected/ineligible metadata:",PLAN["observation_issues"])
else:
    print("No real plan generated: candidate preparation has not completed.")
OBSERVED_ADAPTATION = {"status":"not_evaluable","reason":"No eligible real observed site data supplied"}
if PLAN and (PLAN["counts"]["eligible_unique_accepted_sites"] or PLAN["counts"]["eligible_review_sites"]):
    before=rank_candidates(CANDIDATES,[],CANDIDATE_MANIFEST,CONFIG,now=datetime.fromisoformat(PLAN["as_of"]))
    before_ids={r["candidate_id"] for r in before["recommendations"]};after_ids={r["candidate_id"] for r in PLAN["recommendations"]}
    OBSERVED_ADAPTATION={"status":"descriptive_recalculation_only","before_plan_hash":before["plan_hash"],"after_plan_hash":PLAN["plan_hash"],"changed_recommendations":len(before_ids.symmetric_difference(after_ids)),"warning":"A changed or unchanged plan is not proof of field accuracy, weight validity or learned adaptation."}
print(json.dumps(OBSERVED_ADAPTATION,indent=2))


### Technical tests, separate from observed evidence

The software suite uses synthetic records to test accepted-site adaptation, rejected-scan follow-up, missing/invalid/stale data, repeat-site confirmation/conflicts, weights and component sums, permissions, spacing/caps, version/hash reproducibility, API authentication and body bounds. The separate numerical terrain tests use 24 genuine terrain fixtures plus fixed-CDF/nodata/boundary checks. Neither establishes environmental predictive accuracy.

The sizing probe uses **10,000 synthetic candidates and 100 synthetic observations** in a separate lightweight process. Its memory/latency are local measurements, not guaranteed Render free-instance performance. Test datasets cannot be packaged as production candidates.


In [ ]:
# 7. Run all deterministic engine/API/numerical tests.
import unittest
sys.path.insert(0,str(WORK/"tests"))
stream=io.StringIO()
suite=unittest.defaultTestLoader.discover(str(WORK/"tests"),pattern="test_*.py")
result=unittest.TextTestRunner(stream=stream,verbosity=2).run(suite)
(OUTPUT/"software_tests.txt").write_text(stream.getvalue())
TEST_EVIDENCE={"run":result.testsRun,"failures":len(result.failures),"errors":len(result.errors),"passed":result.wasSuccessful(),"kind":"software_verification_not_scientific_validation"}
print(stream.getvalue());print(json.dumps(TEST_EVIDENCE,indent=2))
if not result.wasSuccessful(): raise AssertionError("Fix failed tests before using the deployment package")


In [ ]:
# 8. Separate-process synthetic runtime sizing.
env=dict(os.environ)
env["PYTHONPATH"]=str(WORK)+os.pathsep+str(WORK/"tests")+os.pathsep+env.get("PYTHONPATH","")
probe=subprocess.run([sys.executable,str(WORK/"tests/benchmark_runtime.py")],check=True,capture_output=True,text=True,env=env)
BENCHMARK=json.loads(probe.stdout)
(OUTPUT/"synthetic_runtime_sizing.json").write_text(json.dumps(BENCHMARK,indent=2))
print(json.dumps(BENCHMARK,indent=2))


In [ ]:
# 9. Export whitelist-only deployment packages. Never includes observations or secrets.
from build_bundle import bundle
EXPORTS=WORK/"exports";EXPORTS.mkdir(exist_ok=True)
BUNDLE=bundle(EXPORTS/"EcoSignal_Adaptive_Planner_Render.zip",include_real_data=DATA_READY)
SITE_ZIP=EXPORTS/"EcoSignal_Adaptive_Planner_Sites_Integration.zip"
with zipfile.ZipFile(SITE_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted((WORK/"site-integration").glob("*")):z.write(p,"site-integration/"+p.name)
    z.write(WORK/"README.md","README.md")
for archive in [Path(BUNDLE["zip"]),SITE_ZIP]:
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        assert not any("reference/" in n or "real_plan" in n or n.endswith(".env") or "observations.json" in n for n in z.namelist())
print(json.dumps(BUNDLE,indent=2))
print("Sites adapter:",SITE_ZIP)
if not DATA_READY: print("UNREADY TEMPLATE ONLY: rerun real candidate preparation before deploying for real recommendations.")


### Deploy on Render, then connect the actual Site

1. Download `EcoSignal_Adaptive_Planner_Render.zip`. Confirm **ready_for_data_serving=true** in the report and nonzero real candidate rows. Put its `EcoSignal_Adaptive_Planner/` contents in a separate GitHub repository; do not commit secrets or observation exports.
2. Create a **Python Web Service** on Render, choose Free if available. Build: `pip install -r requirements.txt`. Start: `python run_server.py`. The included `render.yaml` describes the same service. This service needs no image models, terrain grid, DEM, API key, rasterio or OSMnx at runtime.
3. In Render's environment settings, set a new **`PLANNER_API_TOKEN`**. You can create one privately with `import secrets; print(secrets.token_urlsafe(32))` in a separate private runtime, then clear that cell's output. Do not paste it into this notebook, browser code or GitHub. This is the planner's token, separate from your image-analysis tokens.
4. Check `https://YOUR-PLANNER.onrender.com/health`: `ready:true` and a nonzero `real_candidate_rows`. Missing real data returns an explicit 503 for recommendations.
5. Bring the new Render base URL and generated Sites integration ZIP to the chat managing EcoSignal. The Site needs **`ECOSIGNAL_PLANNER_URL`** and **`ECOSIGNAL_PLANNER_TOKEN`** in its server-side secret settings. Its browser calls same-origin `/api/sampling/recommend`; its server calls Render's `/sampling/recommend` using Bearer authentication. Never ship the token to the browser.
6. Apply `node site-integration/install.mjs /path/to/latest-site-source`, then typecheck, build, test real workflows and publish preserving the existing audience. The installer checks exact compatible anchors and creates a backup; it was verified against v26 in a temporary source copy. A later incompatible source must be adapted, not forced. It replaces the fixed example candidate list, displays live priorities/reasons and prepares a newly selected scan without reusing an old photo.

Free services can sleep and cold-start. Verify current [Render free-service limits](https://render.com/docs/free) before a demonstration. A local benchmark does not guarantee latency/memory on Render; measure the deployed service. Keep slow-request/loading/retry states enabled.

**How to demonstrate adaptation with real data:** fetch recommendations; visit one recommended site; capture an actual photo with its true time/location and confirmed site ID; obtain/confirm its real analysis; refresh the planner with the new observation export. It should count one unique accepted site and exclude nearby accepted locations, or prioritize a review/retake if the analysis was rejected. Compare counts, source hashes, objectives and plan hashes. Do not invent a scan to make the plan change. Only synthetic tests here create artificial records.


In [ ]:
# 10. Optional LIVE planner test after YOU deploy it. No redirects or token logging.
LIVE_EVIDENCE={"status":"not_run","reason":"No live planner deployment supplied/tested"}
if RUN_LIVE_RENDER:
    import urllib.request, urllib.parse, getpass
    base=RENDER_BASE_URL.rstrip("/");parsed=urllib.parse.urlparse(base)
    if parsed.scheme!="https" or not parsed.netloc or parsed.username or parsed.password or parsed.query or parsed.fragment: raise ValueError("Use an HTTPS planner base URL without credentials/query")
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self,*args,**kwargs): return None
    opener=urllib.request.build_opener(NoRedirect)
    with opener.open(base+"/health",timeout=120) as response:health=json.load(response)
    if not health.get("ready") or health.get("real_candidate_rows",0)<=0:raise ValueError("Live planner is not ready with real data/token")
    token=getpass.getpass("Planner server token (hidden; not saved): ")
    try:
        body=json.dumps({"observations":OBSERVATIONS,"limit":10},allow_nan=False).encode()
        req=urllib.request.Request(base+"/sampling/recommend",data=body,headers={"Content-Type":"application/json","Authorization":"Bearer "+token},method="POST")
        with opener.open(req,timeout=120) as response:live=json.load(response)
        if live.get("synthetic_test_only"): raise ValueError("Server returned synthetic-test output")
        LIVE_EVIDENCE={"status":"passed","method_version":live.get("method_version"),"health":health,"counts":live.get("counts"),"plan_hash":live.get("plan_hash")}
        print(json.dumps(LIVE_EVIDENCE,indent=2))
    finally:
        token=None
        if "req" in locals():del req


In [ ]:
# 11. Save the preparation report. Real observation exports remain OUTSIDE deployment ZIPs.
REPORT={"generated_utc":datetime.now(timezone.utc).isoformat(),"method_version":CONFIG.method_version,"profile":CONFIG.profile,"planner_weights":CONFIG.weights,"unified_score_weights_unchanged":{"satellite":.35,"aqi":.15,"lichen_visible_coverage":.25,"terrain":.25,"growth_form":0},"preparation":PREPARATION,"real_input_observation_rows":len(OBSERVATIONS),"eligible_observed_sites":PLAN["counts"]["eligible_unique_accepted_sites"] if PLAN else 0,"observed_adaptation":OBSERVED_ADAPTATION,"numerical_reference_checks":NUMERIC_EVIDENCE,"software_tests":TEST_EVIDENCE,"synthetic_runtime_sizing":BENCHMARK,"render_package":BUNDLE,"live_api":LIVE_EVIDENCE,"scientific_validation":{"status":"not_run","reason":"No independent appropriately defined field ground truth; software checks and planner recalculation are not predictive validation."},"remaining_steps":["Real candidate preparation if unready","Deploy/test new lightweight Render service","Apply adapter to latest Site, configure server-only secrets, verify and publish preserving audience","Collect confirmed dated field observations and independent validation data"]}
REPORT_PATH=EXPORTS/"EcoSignal_Planner_Preparation_Report.json"
REPORT_PATH.write_text(json.dumps(REPORT,indent=2,allow_nan=False))
print("Real candidate readiness:",DATA_READY,"| Eligible observed sites:",REPORT["eligible_observed_sites"])
print("Software tests:",TEST_EVIDENCE,"| Independent scientific validation: not_run")
print("Report:",REPORT_PATH)
if DOWNLOAD_OUTPUTS:
    from google.colab import files
    for p in [Path(BUNDLE["zip"]),SITE_ZIP,REPORT_PATH]: files.download(str(p))


### What is and is not complete

Passing this notebook establishes reproducible technical preparation, numeric terrain sampling, ranking/adaptation behavior on test fixtures, and deployable code. With genuine candidate screening it can generate a real candidate-only plan; with confirmed dated real biological records it can recalculate that plan descriptively. It does **not** create missing observed scans, establish optimal weights, prove ecological accuracy, validate species pollution tolerance, or provide independent environmental ground truth. The frozen 2024 susceptibility reference remains historical and uncalibrated. Full unified-score source/time/geospatial gates remain separate and are not bypassed by the planner.

Reference source copies are preserved in `reference/`; the original Library notebook is unchanged. Do not upload a saved notebook containing token outputs or private observations to a public repository. Reopen this clean copy for sharing; deployment ZIPs are generated from an explicit file whitelist.
